# Student Monthly Spending RL — Training & Analysis Notebook

Notebook referensi untuk melatih, mengevaluasi secara berkala, menyimpan checkpoint terbaik, dan menganalisis alokasi anggaran bulanan agen `MaskablePPO` + `Behavior Cloning` (`StudentPPOAgent`) pada `StudentLifeEnv`.

Kamu dapat mengganti `ENV_CONFIG_PATH` ke template preset (`configs/presets/*.yaml`), root `env_config.yaml`, atau konfigurasi custom milikmu di `configs/custom/*.yaml`.


In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
import stable_baselines3
import sb3_contrib

from src.env import StudentLifeEnv
from src.agents import StudentPPOAgent, set_global_seed

print(f"Python/Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(
    f"Gymnasium: {gym.__version__} | SB3: {stable_baselines3.__version__} | "
    f"SB3-Contrib: {sb3_contrib.__version__}"
)

## 1. Konfigurasi Eksperimen & Validasi Environment

In [ ]:
ENV_CONFIG_PATH = "configs/presets/standard_student.yaml"  # Atau "env_config.yaml" / "configs/custom/<nama>.yaml"
EXPERIMENT_NAME = "standard_student"
RUNS_ROOT_DIR = "runs"

TRAIN_CONFIG = {
    "seed": 42,
    "num_environments": 4,
    "features_dim": 128,
    "hidden_dim": 128,
    "policy_hidden_layers": [128, 128],
    "value_hidden_layers": [128, 128],
    "learning_rate": 1e-4,               # Halus untuk fine-tuning pasca-BC
    "rollout_steps_per_env": 300,       # 300 * 4 env = 1.200 step (8 episode penuh) per update
    "batch_size": 120,
    "num_epochs": 10,
    "discount_factor_gamma": 0.995,      # Matches budget_potential_shaping.gamma in env_config.yaml
    "gae_lambda": 0.98,
    "clip_range": 0.2,
    "entropy_coefficient": 0.015,
    "target_kl": 0.02,                  # Rem KL otomatis mencegah BC policy collapse
    "use_action_masking": True,         # MaskablePPO (sb3-contrib)
    "share_features_extractor": False,  # Pisahkan extractor Actor & Critic
    "device": "auto",
}

TRAINING_SCHEDULE = {
    "total_timesteps": 3_000_000,
    "eval_interval_steps": 100_000,
    "eval_episodes": 10,
    "final_eval_episodes": 20,
    "eval_seed": 1000,
    "survival_phase_timesteps": 1_000_000,   # Target: 100% completion before 500k; afterward optimize leftover balance
}

with open(ENV_CONFIG_PATH, "r", encoding="utf-8") as config_file:
    active_env_config = yaml.safe_load(config_file)

sample_env = StudentLifeEnv(yaml_path=ENV_CONFIG_PATH)
print("Observation space:", sample_env.observation_space)
print("Action space:", sample_env.action_space, "| total mask logits:", sample_env.action_masks().shape[0])
print(
    "Budget bulanan:",
    f"Rp {active_env_config['finance']['monthly_budget']:,.0f}",
    "| Kuota:",
    f"Rp {active_env_config['finance']['fixed_expenses']['mobile_data']:,.0f}",
    "| Saldo operasional tunggal:",
    f"Rp {sample_env.finance.initial_operational_budget_idr:,.0f}",
)

## 2. Training Loop Reproducible & Best-Model Checkpointing

In [ ]:
set_global_seed(TRAIN_CONFIG["seed"])

agent = StudentPPOAgent(
    yaml_path=ENV_CONFIG_PATH,
    **TRAIN_CONFIG,
    verbose=0,
)

bc_summary = agent.pretrain_bc(num_episodes=80, epochs=25, learning_rate=1e-3, entropy_coef=0.03)

run_result = agent.train_with_logging(
    **TRAINING_SCHEDULE,
    experiment_name=EXPERIMENT_NAME,
    runs_root_dir=RUNS_ROOT_DIR,
    progress_bar=False,
)

run_dir = Path(run_result["run_dir"])
eval_history_df = run_result["eval_history_df"]
final_summary = run_result["final_summary"]
final_episodes_df = run_result["final_episodes_df"]
best_episode_trace_df = run_result["best_episode_trace_df"]

print(f"\nRun selesai. Direktori artefak: {run_dir}")
print(f"Best checkpoint pada timestep: {run_result['best_checkpoint_timestep']:,}")
print(
    f"Episode evaluasi terbaik: episode {final_summary['best_eval_episode_index']} "
    f"(seed {final_summary['best_eval_episode_seed']}, reward {final_summary['max_episode_reward']:.2f})"
)

## 2b. Eval best / lanjut training tanpa variabel dari cell training

In [ ]:
from pathlib import Path
import json
import pandas as pd

RUN_DIR = Path(RUN_DIR) if 'RUN_DIR' in globals() else max((p for p in Path(RUNS_ROOT_DIR).glob('*') if (p/'train_config.json').exists()), key=lambda p: p.stat().st_mtime)
run_dir = RUN_DIR
assert (run_dir/'best_model.zip').exists()
eval_history_df = pd.read_csv(run_dir/'eval_history.csv') if (run_dir/'eval_history.csv').exists() else pd.DataFrame()
recovery_agent = StudentPPOAgent.load_from_run(run_dir, checkpoint='best')
final_summary, final_episodes_df, best_episode_trace_df = recovery_agent.evaluate_detailed(num_episodes=20, deterministic=True, eval_seed=11000, record_best_episode_trace=True)
final_summary['eval_seed'] = 11000
(run_dir/'manual_eval_summary.json').write_text(json.dumps(final_summary, indent=2), encoding='utf-8')
final_episodes_df.to_csv(run_dir/'manual_eval_episodes.csv', index=False)
best_episode_trace_df.to_csv(run_dir/'manual_best_episode_trace.csv', index=False)
print(f"completion={final_summary['completion_rate']:.0%} | hari={final_summary['mean_days_survived']:.1f}/30 | sisa=Rp {final_summary['mean_final_balance_idr']:,.0f}")

# Resume: ganti blok evaluasi di atas dengan baris di bawah jika ingin lanjut training.
# agent = StudentPPOAgent(yaml_path=str(RUN_DIR/'env_config_snapshot.yaml'), **TRAIN_CONFIG, verbose=0)
# run_result = agent.train_with_logging(**TRAINING_SCHEDULE, experiment_name=EXPERIMENT_NAME, runs_root_dir=RUNS_ROOT_DIR, progress_bar=False, resume_from_run=RUN_DIR, resume_checkpoint='best')
# run_dir, eval_history_df, final_summary, final_episodes_df, best_episode_trace_df = (run_result[k] for k in ('run_dir','eval_history_df','final_summary','final_episodes_df','best_episode_trace_df'))

## 3. Ringkasan Metrik Evaluasi & Alokasi Anggaran 1 Bulan (Best Checkpoint)

In [ ]:
summary_table = pd.DataFrame(
    list(final_summary.items()),
    columns=["metric", "value"],
)
display(summary_table)

# Tabel alokasi pengeluaran 1 bulan (pedoman distribusi budget hasil RL)
initial_budget = float(sample_env.finance.initial_operational_budget_idr)
allocation_rows = [
    ("transport", final_summary["mean_spent_transport_idr"]),
    ("grocery_cooking", final_summary["mean_spent_grocery_idr"]),
    ("external_food_warung_snack", final_summary["mean_spent_external_food_idr"]),
    ("cafe", final_summary["mean_spent_cafe_idr"]),
    ("minimarket_impulse", final_summary["mean_spent_minimarket_impulse_idr"]),
    ("remaining_balance", final_summary["mean_final_balance_idr"]),
]
allocation_df = pd.DataFrame(allocation_rows, columns=["category", "mean_idr"])
allocation_df["share_of_operational_budget_pct"] = (
    allocation_df["mean_idr"] / initial_budget * 100.0
)
display(allocation_df.round({"mean_idr": 0, "share_of_operational_budget_pct": 2}))

preferred_episode_cols = [
    "episode_index",
    "episode_reward",
    "episode_steps",
    "termination_reason",
    "class_attendance_rate",
    "final_balance_idr",
    "spent_transport_idr",
    "spent_grocery_idr",
    "spent_external_food_idr",
    "spent_cafe_idr",
    "mean_energy",
    "mean_hunger",
    "mean_mood",
    "mean_academic_stress",
    "total_spoiled_units",
    "grocery_shop_count",
    "cook_fresh_count",
    "meal_prep_count",
    "reheat_prep_count",
    "pack_bekal_count",
    "eat_bekal_count",
    "hub_or_leisure_count",
    "cafe_session_count",
    "night_cramming_count",
]
available_episode_cols = [c for c in preferred_episode_cols if c in final_episodes_df.columns]
display(final_episodes_df[available_episode_cols])

## 4. Visualisasi Progres Evaluasi & Trace Episode Terbaik (30 Hari)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(eval_history_df["timesteps"], eval_history_df["mean_episode_reward"], marker="o")
axes[0].fill_between(
    eval_history_df["timesteps"],
    eval_history_df["mean_episode_reward"] - eval_history_df["std_episode_reward"],
    eval_history_df["mean_episode_reward"] + eval_history_df["std_episode_reward"],
    alpha=0.2,
)
axes[0].set_title("mean_episode_reward vs timesteps")
axes[0].set_xlabel("timesteps")
axes[0].set_ylabel("mean_episode_reward")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    eval_history_df["timesteps"],
    eval_history_df["completion_rate"] * 100.0,
    marker="o",
    label="completion_rate (%)",
)
axes[1].plot(
    eval_history_df["timesteps"],
    eval_history_df["mean_class_attendance_rate"] * 100.0,
    marker="s",
    label="mean_class_attendance_rate (%)",
)
axes[1].set_title("completion_rate & mean_class_attendance_rate")
axes[1].set_xlabel("timesteps")
axes[1].set_ylabel("percentage (%)")
axes[1].set_ylim(-5, 105)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(
    eval_history_df["timesteps"],
    eval_history_df["mean_final_balance_idr"] / 1_000.0,
    marker="o",
    label="mean_final_balance_idr (ribu Rp)",
)
axes[2].plot(
    eval_history_df["timesteps"],
    eval_history_df["mean_spent_grocery_idr"] / 1_000.0,
    marker="s",
    label="mean_spent_grocery_idr (ribu Rp)",
)
axes[2].plot(
    eval_history_df["timesteps"],
    eval_history_df["mean_spent_external_food_idr"] / 1_000.0,
    marker="^",
    label="mean_spent_external_food_idr (ribu Rp)",
)
axes[2].set_title("balance & food spending vs timesteps")
axes[2].set_xlabel("timesteps")
axes[2].set_ylabel("ribu Rp")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if not best_episode_trace_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    for column_name in ["energy", "hunger", "mood", "academic_stress", "cafe_craving"]:
        axes[0].plot(best_episode_trace_df["step"], best_episode_trace_df[column_name], label=column_name)
    axes[0].set_title("biometrics_state per step (best evaluation episode)")
    axes[0].set_ylabel("score (0-100)")
    axes[0].set_ylim(-5, 105)
    axes[0].legend(ncol=5, loc="upper right")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(
        best_episode_trace_df["step"],
        best_episode_trace_df["balance_idr"] / 1_000.0,
        label="balance_idr (ribu Rp)",
    )
    axes[1].set_title("balance_idr per step (best evaluation episode)")
    axes[1].set_xlabel("step (0-149)")
    axes[1].set_ylabel("ribu Rp")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 5. Trace Detail Episode Terbaik (Hari 1 s/d Hari 30: 5 Step per Hari)

In [ ]:
if "run_dir" not in globals():
    run_folders = sorted(
        [path for path in Path(RUNS_ROOT_DIR).iterdir() if path.is_dir()],
        key=lambda path: path.name,
    )
    assert run_folders, "Belum ada folder run di dalam direktori runs/."
    run_dir = run_folders[-1]

best_trace_csv_path = run_dir / "best_episode_trace.csv"
if best_trace_csv_path.exists():
    best_episode_trace_df = pd.read_csv(best_trace_csv_path)
else:
    episodes_table = pd.read_csv(run_dir / "final_eval_episodes.csv")
    best_episode_row = episodes_table.loc[episodes_table["episode_reward"].idxmax()]
    loaded_best_agent = StudentPPOAgent.load_from_run(run_dir, checkpoint="best")
    best_episode_trace_df = loaded_best_agent.record_episode_trace(
        episode_seed=int(best_episode_row["episode_seed"]),
        deterministic=True,
    )
    best_episode_trace_df.to_csv(best_trace_csv_path, index=False)

print(
    f"Run: {run_dir.name} | "
    f"Seed episode terbaik: {int(best_episode_trace_df['episode_seed'].iloc[0])} | "
    f"Total step: {len(best_episode_trace_df)} | "
    f"Total reward: {best_episode_trace_df['cumulative_reward'].iloc[-1]:.2f}"
)

detailed_columns = [
    "day",
    "day_name",
    "time_slot",
    "hours",
    "class_scheduled",
    "location_before",
    "location_after",
    "executed_route",
    "executed_activity",
    "executed_meal_mode",
    "meal_signature",
    "purchased_groceries",
    "step_spend_idr",
    "balance_idr",
    "energy",
    "hunger",
    "mood",
    "academic_stress",
    "cafe_craving",
    "rice_stock",
    "egg_stock",
    "chicken_stock",
    "tempe_stock",
    "meal_prep_stock",
    "packed_lunch_stock",
    "events_summary",
    "step_reward",
    "cumulative_reward",
]

with pd.option_context(
    "display.max_rows", 160,
    "display.max_columns", 40,
    "display.width", 280,
    "display.max_colwidth", 65,
):
    display(best_episode_trace_df[detailed_columns].round({
        "step_spend_idr": 0,
        "balance_idr": 0,
        "energy": 1,
        "hunger": 1,
        "mood": 1,
        "academic_stress": 1,
        "cafe_craving": 1,
        "step_reward": 2,
        "cumulative_reward": 2,
    }))

daily_summary_df = (
    best_episode_trace_df.groupby(["day", "day_name"], sort=True)
    .agg(
        total_spend_idr=("step_spend_idr", "sum"),
        end_balance_idr=("balance_idr", "last"),
        routes_used=("executed_route", lambda s: ", ".join(r for r in s if r != "stay") or "stay"),
        activities=("executed_activity", lambda s: " -> ".join(s)),
        meals=("executed_meal_mode", lambda s: ", ".join(m for m in s if m != "skip") or "skip"),
        mean_energy=("energy", "mean"),
        mean_mood=("mood", "mean"),
        mean_stress=("academic_stress", "mean"),
        daily_reward=("step_reward", "sum"),
    )
    .reset_index()
)

with pd.option_context("display.max_rows", 35, "display.max_columns", 20, "display.width", 260, "display.max_colwidth", 90):
    display(daily_summary_df.round({
        "total_spend_idr": 0,
        "end_balance_idr": 0,
        "mean_energy": 1,
        "mean_mood": 1,
        "mean_stress": 1,
        "daily_reward": 2,
    }))

## 6. Perbandingan Antar-Eksperimen & Reproduksi dari Snapshot Run

In [ ]:
master_summary_path = Path(RUNS_ROOT_DIR) / "experiments_summary.csv"
if master_summary_path.exists():
    experiments_df = pd.read_csv(master_summary_path)
    preferred_cols = [
        "run_id",
        "experiment_name",
        "env_config_sha256",
        "use_action_masking",
        "seed",
        "total_timesteps",
        "best_checkpoint_timestep",
        "mean_episode_reward",
        "completion_rate",
        "mean_class_attendance_rate",
        "mean_final_balance_idr",
        "mean_spent_transport_idr",
        "mean_spent_grocery_idr",
        "mean_spent_external_food_idr",
        "mean_spent_cafe_idr",
        "mean_cook_fresh_count",
        "mean_eat_bekal_count",
        "mean_spoiled_units",
    ]
    available_cols = [col for col in preferred_cols if col in experiments_df.columns]
    display(experiments_df[available_cols].sort_values(by="mean_episode_reward", ascending=False))

reloaded_agent = StudentPPOAgent.load_from_run(run_dir, checkpoint="best")
recheck_summary, _, _ = reloaded_agent.evaluate_detailed(
    num_episodes=TRAINING_SCHEDULE["final_eval_episodes"],
    deterministic=True,
    eval_seed=TRAINING_SCHEDULE["eval_seed"] + 10_000,
)
print(
    "Verifikasi reload dari snapshot:",
    f"original={final_summary['mean_episode_reward']:.4f}",
    f"| reloaded={recheck_summary['mean_episode_reward']:.4f}",
)